**Eksperyment 2: Wpływ funkcji straty na aproksymację funkcji w zadaniu regresji**
**Testowane funkcje:**
- MSE
- Huber
**Każdy eksperyment będzie miał taką samą architektóre sieci tzn:**
- 1 hidden layer
- 16 perceptronów per layer
- aktywacja wyjściowa to f(x) = x
- funkcja aktywacji to sigmoid
- optymalizatior to SGD
- learning rate = 0.001
- ilość epok = 500
- batch = 32

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from traitlets.config import catch_config_error

from mlp_model.activations.activation import Activation
from mlp_model.activations.relu import ReLU
from mlp_model.activations.swish import Swish

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.losses.loss import Loss
from mlp_model.losses.huber_loss import HuberLoss
from mlp_model.mlp import MLP
from mlp_model.optimizers.sgd import SGDOptimizer
from trainer.trainer import MLPTrainer
from sklearn.model_selection import train_test_split
from visualisation.regression_visualise import plot_learning_curve, plot_regression_fit, \
    plot_weights_evolution, print_cv_summary

data_kind = "activation"
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/regression" / f"data.{data_kind}.train.{data_count}.csv"
df = pd.read_csv(train_filepath)
x = df["x"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
y = df["y"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)

C:\Users\arczi\PycharmProjects\NeuralNetworks\visualize_plot\helo_helo.png


In [5]:
lr = 0.005
epochs = 100
folds = 5
layers =     [
        Hidden(1, 16, Sigmoid()),
        OutputLayer(16, 1)
    ]
batch_size = 32
test_size=0.2
random_state=42

In [6]:
def cross_validation(loss: Loss, filename: str | None):
    mlp = MLP(layers=layers)
    mlp.initialize_weights(XavierInitializer())
    sgd = SGDOptimizer(mlp, lr)
    trainer = MLPTrainer(mlp, sgd, loss, epochs, batch_size)
    summary = trainer.cross_validate(x, y, folds)
    print_cv_summary(summary, filename)

def visualization(loss: Loss, filename: str | None):
    mlp = MLP(layers=layers)
    mlp.initialize_weights(XavierInitializer())
    sgd = SGDOptimizer(mlp, lr)
    trainer = MLPTrainer(mlp, sgd, loss, epochs, batch_size)
    X_train, X_val, y_train, y_val = train_test_split(
        x,
        y,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )
    trainer.fit(X_train, y_train, X_val, y_val)
    plot_learning_curve(trainer.history, filename)
    plot_regression_fit(trainer.history, mlp, X_train, y_train, X_val, y_val, filename)
    for i in range(len(layers)):
        plot_weights_evolution(trainer.history, i,  16, filename)


def test_loss(loss: Loss, filename: str = None):
    cross_validation(loss, filename)
    visualization(loss, filename)

In [7]:
test_loss(MSELoss(), "/regression/loss/MSELoss/mse");
deltas = [1, 15]
for delta in deltas:
    test_loss(HuberLoss(delta), f"/regression/loss/HuberLoss-{delta}/huber");

Fold 1: loss = 0.205026 | metrics {'r2': 0.9999731540398774, 'mae': 0.2949644211649796, 'mse': 0.20502610526840268}
Fold 2: loss = 0.211286 | metrics {'r2': 0.9999725874055527, 'mae': 0.29227575088859653, 'mse': 0.21128581025597054}
Fold 3: loss = 0.316859 | metrics {'r2': 0.9999575013587941, 'mae': 0.34703761685458656, 'mse': 0.3168594372180808}
Fold 4: loss = 0.359951 | metrics {'r2': 0.9999569279753853, 'mae': 0.41442308982158954, 'mse': 0.3599509465030819}
Fold 5: loss = 0.468752 | metrics {'r2': 0.9999439432051807, 'mae': 0.42367393893431016, 'mse': 0.46875213173193403}
---------------------------------------------------------------
Cross validation with 5 folds:
Mean loss: 0.312375 (std: 0.098468)
Mean metrics:
  R2: 0.999961 (std: 0.000011)
  MAE: 0.354475 (std: 0.056299)
  MSE: 0.312375 (std: 0.098468)
---------------------------------------------------------------
Fold 1: loss = 47.318216 | metrics {'r2': 0.2101420854656072, 'mae': 47.810618547336915, 'mse': 6032.248099627933}